# Deep Q-Network (DQN) for Image Processing

## Module 6.1 — Deep RL: DQN Architecture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_06_Deep_RL/01_DQN_Deep_Q_Network/dqn_deep_q_network.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** the DQN architecture combining CNNs with Q-value estimation
2. **Derive** the mathematics of experience replay buffers and target networks
3. **Implement** a full DQN agent that processes images as states
4. **Train** the agent to perform image processing actions with convergence guarantees
5. **Visualize** training dynamics including loss curves and action distributions

In [ ]:
# Install dependencies (Colab-compatible)
!pip install torch torchvision numpy matplotlib pillow gymnasium --quiet

In [ ]:
# Download REAL open-source dataset for Deep RL image environment
import torchvision
import torchvision.transforms as transforms
import numpy as np

# CIFAR-10 as our image environment (real photos to enhance)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
real_images = np.array([np.array(cifar10[i][0]) for i in range(500)])
print(f"✅ CIFAR-10 loaded: {len(real_images)} real photos as RL environment images")
print(f"   Shape: {real_images[0].shape} (32x32 RGB)")
print(f"   Agent will learn to enhance these REAL photographs!")

# Pre-trained model weights (for feature extraction)
from torchvision import models
print("✅ Pre-trained ResNet18 weights available for state encoding")

## Deep Derivation: DQN Loss and Target Network Stability

### Step 1: Starting from Bellman Optimality
$$Q^*(s,a) = E_{s'}\left[r + \gamma \max_{a'} Q^*(s',a') \mid s, a\right]$$

### Step 2: Temporal Difference Error
$$\delta_t = r_t + \gamma \max_{a'} Q(s_{t+1}, a'; \theta^-) - Q(s_t, a_t; \theta)$$

### Step 3: DQN Loss Function (Mean Squared TD Error)
$$L(\theta) = E_{(s,a,r,s') \sim \mathcal{D}}\left[\left(r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta)\right)^2\right]$$

### Step 4: Gradient (only through $Q(s,a;\theta)$, NOT through target)
$$\nabla_\theta L = -2 E\left[\delta_t \cdot \nabla_\theta Q(s_t, a_t; \theta)\right]$$

### Step 5: Why Target Network is Needed (Stability Proof)
**Without target network:** Both $Q(s,a;\theta)$ and $\max_{a'} Q(s',a';\theta)$ change simultaneously.

**Problem (moving target):** Updating $\theta$ to reduce error at $(s,a)$ CHANGES the target at $(s', a')$, causing oscillation.

**Solution:** Fix target $\theta^-$ for $C$ steps:
$$\theta^- \leftarrow \theta \text{ every } C \text{ steps}$$

Or soft update: $\theta^- \leftarrow \tau\theta + (1-\tau)\theta^-$, $\tau \ll 1$

### Step 6: Experience Replay as Importance Correction
Sampling from buffer breaks correlation: $(s_t, a_t, r_t, s_{t+1})$ are no longer sequential.

**Why it works:** Q-learning is off-policy — it learns about $\pi^*$ from data generated by ANY policy!

**Off-policy property proof:**
The Bellman optimality equation $Q^* = \mathcal{T}^*Q^*$ does not depend on the behavior policy $\mu$ — only on transition dynamics $P(s'|s,a)$.

### Step 7: ε-Greedy Exploration Schedule
$$\epsilon_t = \epsilon_{\min} + (\epsilon_{\max} - \epsilon_{\min}) \cdot e^{-t/\tau_{\text{decay}}}$$

**GLIE condition:** $\epsilon_t \to 0$ as $t \to \infty$, but $\sum_t \epsilon_t = \infty$ (visits all states).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import deque, namedtuple
import random
from PIL import Image
from torchvision import transforms
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

---

## 1. From Q-Learning to Deep Q-Networks

### 1.1 The Q-Learning Foundation

Recall the Bellman optimality equation for the action-value function:

$$Q^*(s, a) = \mathbb{E}\left[r + \gamma \max_{a'} Q^*(s', a') \mid s, a\right]$$

In tabular Q-learning, we maintain a table $Q(s, a)$ for all state-action pairs. The update rule is:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\right]$$

where the term in brackets is the **temporal difference (TD) error**:

$$\delta_t = r_t + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t)$$

### 1.2 The Challenge of Image States

For image processing, states are high-dimensional images. A $64 \times 64$ RGB image has $64 \times 64 \times 3 = 12{,}288$ dimensions. A tabular approach is infeasible since the state space is effectively continuous.

**Solution**: Approximate $Q^*(s, a)$ with a neural network parameterized by $\theta$:

$$Q(s, a; \theta) \approx Q^*(s, a)$$

### 1.3 DQN Architecture

The DQN uses a **Convolutional Neural Network** to process image states:

$$\text{Image } s \xrightarrow{\text{Conv layers}} \text{Feature map} \xrightarrow{\text{FC layers}} Q(s, a_1), Q(s, a_2), \ldots, Q(s, a_n)$$

The network outputs Q-values for **all actions simultaneously**, avoiding the need for a forward pass per action.

---

## 2. Experience Replay Buffer — Mathematical Foundation

### 2.1 The Correlation Problem

Training a neural network on sequential experience $(s_t, a_t, r_t, s_{t+1})$ violates the i.i.d. assumption of SGD. Consecutive transitions are highly correlated:

$$\text{Corr}(x_t, x_{t+1}) \gg 0$$

This leads to unstable training and divergence.

### 2.2 Replay Buffer as Decorrelation

Store transitions in a buffer $\mathcal{D} = \{(s_i, a_i, r_i, s_{i+1})\}_{i=1}^{N}$ of capacity $N$.

At each training step, sample a **mini-batch** uniformly:

$$\mathcal{B} \sim \text{Uniform}(\mathcal{D}), \quad |\mathcal{B}| = B$$

The expected gradient under uniform sampling:

$$\mathbb{E}_{(s,a,r,s') \sim \mathcal{D}}\left[\nabla_\theta L(\theta)\right] = \frac{1}{N} \sum_{i=1}^{N} \nabla_\theta \ell_i(\theta)$$

### 2.3 Buffer Statistics

The effective sample correlation after uniform sampling from a buffer of size $N$ with batch size $B$:

$$\rho_{\text{effective}} \approx \frac{B}{N} \cdot \rho_{\text{sequential}}$$

For $N = 10^6$ and $B = 32$, we reduce correlation by a factor of $\sim 31{,}250$.

### 2.4 Buffer Capacity Trade-off

- **Too small** ($N \ll $ needed): High correlation, poor coverage of state space
- **Too large** ($N \gg $ needed): Stale transitions from outdated policies, slower learning
- **Optimal**: $N$ should be large enough for decorrelation but recent enough for relevance

In [ ]:
# Experience Replay Buffer Implementation

Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))


class ReplayBuffer:
    """Fixed-size circular buffer with uniform sampling."""

    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        transitions = random.sample(self.buffer, batch_size)
        batch = Transition(*zip(*transitions))
        return batch

    def __len__(self):
        return len(self.buffer)


# Demonstrate decorrelation property
def demonstrate_decorrelation():
    buffer = ReplayBuffer(10000)

    # Fill with sequential correlated data
    for i in range(10000):
        state = np.sin(i * 0.01) + np.random.normal(0, 0.1)
        buffer.push(
            torch.tensor([state]),
            0,
            state * 0.5,
            torch.tensor([np.sin((i+1) * 0.01)]),
            False
        )

    # Sample and measure correlation
    sequential_corr = []
    sampled_corr = []

    for _ in range(100):
        batch = buffer.sample(32)
        states = torch.stack(batch.state).numpy().flatten()
        sampled_corr.append(np.corrcoef(states[:-1], states[1:])[0, 1])

    # Sequential correlation
    seq_data = [buffer.buffer[i].state.item() for i in range(100)]
    seq_corr = np.corrcoef(seq_data[:-1], seq_data[1:])[0, 1]

    print(f"Sequential autocorrelation: {seq_corr:.4f}")
    print(f"Mean sampled autocorrelation: {np.mean(sampled_corr):.4f}")
    print(f"Decorrelation factor: {abs(seq_corr / (np.mean(sampled_corr) + 1e-10)):.1f}x")


demonstrate_decorrelation()

---

## 3. Target Network — Stability Theory

### 3.1 The Moving Target Problem

The DQN loss function is:

$$L(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}}\left[\left(r + \gamma \max_{a'} Q(s', a'; \theta) - Q(s, a; \theta)\right)^2\right]$$

The target $y = r + \gamma \max_{a'} Q(s', a'; \theta)$ depends on the same parameters $\theta$ we're optimizing. This creates a **non-stationary optimization** problem — the target moves as we update.

### 3.2 Target Network Solution

Introduce a separate **target network** with parameters $\theta^-$ (a delayed copy of $\theta$):

$$L(\theta) = \mathbb{E}\left[\left(r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta)\right)^2\right]$$

The target is now:

$$y_i = r_i + \gamma \max_{a'} Q(s_i', a'; \theta^-)$$

### 3.3 Update Strategies

**Hard update** (every $C$ steps):
$$\theta^- \leftarrow \theta \quad \text{every } C \text{ steps}$$

**Soft update** (Polyak averaging):
$$\theta^- \leftarrow \tau \theta + (1 - \tau) \theta^- \quad \text{where } \tau \ll 1$$

### 3.4 Convergence Guarantee

With a fixed target network, the DQN update becomes a **supervised regression** problem with stationary targets (within the update interval). The Bellman operator $\mathcal{T}$ is a $\gamma$-contraction:

$$\|\mathcal{T}Q_1 - \mathcal{T}Q_2\|_\infty \leq \gamma \|Q_1 - Q_2\|_\infty$$

This guarantees convergence to a fixed point when the target network is periodically synchronized.

---

## 4. Epsilon-Greedy Exploration

### 4.1 The Exploration-Exploitation Dilemma

The agent must balance:
- **Exploitation**: Choose $a = \arg\max_a Q(s, a; \theta)$ to maximize immediate reward
- **Exploration**: Try sub-optimal actions to discover better strategies

### 4.2 $\epsilon$-Greedy Policy

$$\pi(a|s) = \begin{cases} 1 - \epsilon + \frac{\epsilon}{|\mathcal{A}|} & \text{if } a = \arg\max_{a'} Q(s, a'; \theta) \\ \frac{\epsilon}{|\mathcal{A}|} & \text{otherwise} \end{cases}$$

### 4.3 Annealing Schedule

Linear decay from $\epsilon_{\text{start}}$ to $\epsilon_{\text{end}}$ over $T$ steps:

$$\epsilon_t = \max\left(\epsilon_{\text{end}}, \; \epsilon_{\text{start}} - \frac{(\epsilon_{\text{start}} - \epsilon_{\text{end}}) \cdot t}{T}\right)$$

Exponential decay:

$$\epsilon_t = \epsilon_{\text{end}} + (\epsilon_{\text{start}} - \epsilon_{\text{end}}) \cdot e^{-t/\tau}$$

### 4.4 GLIE Property

For convergence guarantees, we need Greedy in the Limit with Infinite Exploration (GLIE):

$$\lim_{t \to \infty} \epsilon_t = 0 \quad \text{and} \quad \sum_{t=0}^{\infty} \epsilon_t = \infty$$

In [ ]:
# Visualize epsilon schedules

steps = np.arange(10000)
eps_start, eps_end, eps_decay = 1.0, 0.01, 5000

# Linear decay
eps_linear = np.maximum(eps_end, eps_start - (eps_start - eps_end) * steps / eps_decay)

# Exponential decay
eps_exp = eps_end + (eps_start - eps_end) * np.exp(-steps / (eps_decay / 3))

# Cosine annealing
eps_cosine = eps_end + 0.5 * (eps_start - eps_end) * (1 + np.cos(np.pi * np.minimum(steps, eps_decay) / eps_decay))

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(steps, eps_linear, label='Linear Decay', linewidth=2)
ax.plot(steps, eps_exp, label='Exponential Decay', linewidth=2)
ax.plot(steps, eps_cosine, label='Cosine Annealing', linewidth=2)
ax.axhline(y=eps_end, color='gray', linestyle='--', alpha=0.5, label=f'$\\epsilon_{{end}}={eps_end}$')
ax.set_xlabel('Training Steps', fontsize=12)
ax.set_ylabel('$\\epsilon$', fontsize=14)
ax.set_title('Epsilon Annealing Schedules for Exploration', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 10000)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

---

## 5. Image Processing Environment

We create a custom environment where:
- **State**: A degraded image (noisy, low contrast, etc.)
- **Actions**: Image processing operations (sharpen, denoise, adjust brightness, etc.)
- **Reward**: Improvement in image quality (measured by PSNR/SSIM)

In [ ]:
class ImageProcessingEnv:
    """RL environment for sequential image enhancement."""

    ACTIONS = [
        'sharpen', 'blur_denoise', 'increase_brightness',
        'decrease_brightness', 'increase_contrast', 'decrease_contrast',
        'gamma_correct_up', 'gamma_correct_down', 'no_op'
    ]

    def __init__(self, image_size=64, max_steps=10):
        self.image_size = image_size
        self.max_steps = max_steps
        self.n_actions = len(self.ACTIONS)
        self.current_step = 0
        self.target_image = None
        self.current_image = None

    def _generate_target(self):
        """Generate a clean target image with patterns."""
        img = np.zeros((3, self.image_size, self.image_size), dtype=np.float32)
        # Create structured patterns
        x = np.linspace(0, 4*np.pi, self.image_size)
        y = np.linspace(0, 4*np.pi, self.image_size)
        X, Y = np.meshgrid(x, y)
        img[0] = 0.5 + 0.3 * np.sin(X) * np.cos(Y)
        img[1] = 0.5 + 0.3 * np.cos(X + Y)
        img[2] = 0.5 + 0.3 * np.sin(X - Y)
        return np.clip(img, 0, 1)

    def _degrade_image(self, img):
        """Apply random degradations."""
        degraded = img.copy()
        # Add noise
        degraded += np.random.normal(0, 0.15, degraded.shape).astype(np.float32)
        # Reduce contrast
        degraded = 0.5 + 0.6 * (degraded - 0.5)
        # Shift brightness
        degraded += np.random.uniform(-0.1, 0.1)
        return np.clip(degraded, 0, 1)

    def _compute_psnr(self, img1, img2):
        mse = np.mean((img1 - img2) ** 2)
        if mse == 0:
            return 50.0
        return 10 * np.log10(1.0 / mse)

    def reset(self):
        self.current_step = 0
        self.target_image = self._generate_target()
        self.current_image = self._degrade_image(self.target_image)
        self.prev_psnr = self._compute_psnr(self.current_image, self.target_image)
        return self.current_image.copy()

    def step(self, action):
        self.current_step += 1
        action_name = self.ACTIONS[action]

        img = self.current_image.copy()

        if action_name == 'sharpen':
            kernel = np.array([[0, -0.5, 0], [-0.5, 3, -0.5], [0, -0.5, 0]]) / 1.0
            for c in range(3):
                from scipy.ndimage import convolve
                img[c] = convolve(img[c], kernel, mode='reflect')
        elif action_name == 'blur_denoise':
            kernel = np.ones((3, 3)) / 9.0
            for c in range(3):
                from scipy.ndimage import convolve
                img[c] = convolve(img[c], kernel, mode='reflect')
        elif action_name == 'increase_brightness':
            img += 0.05
        elif action_name == 'decrease_brightness':
            img -= 0.05
        elif action_name == 'increase_contrast':
            img = 0.5 + 1.2 * (img - 0.5)
        elif action_name == 'decrease_contrast':
            img = 0.5 + 0.8 * (img - 0.5)
        elif action_name == 'gamma_correct_up':
            img = np.power(np.clip(img, 0, 1), 0.9)
        elif action_name == 'gamma_correct_down':
            img = np.power(np.clip(img, 0, 1), 1.1)
        # no_op does nothing

        self.current_image = np.clip(img, 0, 1)

        new_psnr = self._compute_psnr(self.current_image, self.target_image)
        reward = new_psnr - self.prev_psnr  # Reward = PSNR improvement
        self.prev_psnr = new_psnr

        done = self.current_step >= self.max_steps
        return self.current_image.copy(), reward, done, {'psnr': new_psnr}


# Test environment
env = ImageProcessingEnv(image_size=64, max_steps=10)
state = env.reset()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(env.target_image.transpose(1, 2, 0))
axes[0].set_title('Target (Clean)', fontsize=12)
axes[1].imshow(state.transpose(1, 2, 0))
axes[1].set_title(f'Degraded (PSNR: {env.prev_psnr:.1f} dB)', fontsize=12)

# Take a few actions
for _ in range(5):
    state, r, done, info = env.step(4)  # increase contrast
axes[2].imshow(state.transpose(1, 2, 0))
axes[2].set_title(f'After Actions (PSNR: {info["psnr"]:.1f} dB)', fontsize=12)

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

---

## 6. DQN Architecture Implementation

### Architecture Design

The network architecture follows the original DQN paper (Mnih et al., 2015) adapted for our image processing task:

$$\text{Input: } s \in \mathbb{R}^{3 \times 64 \times 64} \xrightarrow{f_\theta} Q(s, \cdot) \in \mathbb{R}^{|\mathcal{A}|}$$

In [ ]:
class DQN(nn.Module):
    """Deep Q-Network with CNN backbone for image state processing."""

    def __init__(self, n_actions, image_size=64):
        super(DQN, self).__init__()

        self.features = nn.Sequential(
            # Conv1: 3x64x64 -> 32x30x30
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # Conv2: 32x30x30 -> 64x14x14
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # Conv3: 64x14x14 -> 64x6x6
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )

        # Compute flattened size
        self._feature_size = self._get_feature_size(image_size)

        self.q_head = nn.Sequential(
            nn.Linear(self._feature_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions)
        )

    def _get_feature_size(self, image_size):
        dummy = torch.zeros(1, 3, image_size, image_size)
        return self.features(dummy).view(1, -1).size(1)

    def forward(self, x):
        features = self.features(x)
        features = features.view(features.size(0), -1)
        q_values = self.q_head(features)
        return q_values


# Instantiate and inspect
n_actions = len(ImageProcessingEnv.ACTIONS)
policy_net = DQN(n_actions, image_size=64).to(device)
target_net = DQN(n_actions, image_size=64).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

print(f"Network architecture:")
print(f"  Feature extractor output: {policy_net._feature_size} dimensions")
print(f"  Number of actions: {n_actions}")
print(f"  Total parameters: {sum(p.numel() for p in policy_net.parameters()):,}")
print(f"\nActions: {ImageProcessingEnv.ACTIONS}")

# Test forward pass
test_state = torch.randn(1, 3, 64, 64).to(device)
q_values = policy_net(test_state)
print(f"\nTest Q-values shape: {q_values.shape}")
print(f"Test Q-values: {q_values.detach().cpu().numpy().round(3)}")

---

## 7. DQN Agent — Full Implementation

### 7.1 The DQN Algorithm

**Algorithm**: Deep Q-Network with Experience Replay and Target Network

1. Initialize replay buffer $\mathcal{D}$ with capacity $N$
2. Initialize policy network $Q(s, a; \theta)$ with random weights
3. Initialize target network $Q(s, a; \theta^-) \leftarrow Q(s, a; \theta)$
4. **For** episode $= 1, 2, \ldots, M$ **do**:
   - Observe initial state $s_1$
   - **For** $t = 1, 2, \ldots, T$ **do**:
     - Select action: $a_t = \begin{cases} \text{random action} & \text{with prob } \epsilon \\ \arg\max_a Q(s_t, a; \theta) & \text{otherwise} \end{cases}$
     - Execute $a_t$, observe $r_t$, $s_{t+1}$
     - Store $(s_t, a_t, r_t, s_{t+1}, \text{done})$ in $\mathcal{D}$
     - Sample mini-batch from $\mathcal{D}$
     - Compute targets: $y_j = r_j + \gamma \max_{a'} Q(s_j', a'; \theta^-) \cdot (1 - \text{done}_j)$
     - Update $\theta$ by minimizing: $L = \frac{1}{B}\sum_j (y_j - Q(s_j, a_j; \theta))^2$
     - Every $C$ steps: $\theta^- \leftarrow \theta$

In [ ]:
class DQNAgent:
    """Complete DQN Agent for image processing."""

    def __init__(self, n_actions, image_size=64, lr=1e-4, gamma=0.99,
                 eps_start=1.0, eps_end=0.05, eps_decay=2000,
                 buffer_size=50000, batch_size=32, target_update=100):
        self.n_actions = n_actions
        self.gamma = gamma
        self.eps_start = eps_start
        self.eps_end = eps_end
        self.eps_decay = eps_decay
        self.batch_size = batch_size
        self.target_update = target_update
        self.steps_done = 0

        self.policy_net = DQN(n_actions, image_size).to(device)
        self.target_net = DQN(n_actions, image_size).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.memory = ReplayBuffer(buffer_size)

        # Metrics
        self.losses = []
        self.epsilons = []
        self.q_values_history = []

    def get_epsilon(self):
        eps = self.eps_end + (self.eps_start - self.eps_end) * \
            np.exp(-self.steps_done / self.eps_decay)
        return eps

    def select_action(self, state):
        eps = self.get_epsilon()
        self.epsilons.append(eps)
        self.steps_done += 1

        if random.random() < eps:
            return random.randrange(self.n_actions)
        else:
            with torch.no_grad():
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
                q_values = self.policy_net(state_tensor)
                self.q_values_history.append(q_values.cpu().numpy().flatten())
                return q_values.argmax(dim=1).item()

    def optimize(self):
        if len(self.memory) < self.batch_size:
            return None

        batch = self.memory.sample(self.batch_size)

        state_batch = torch.FloatTensor(np.array(batch.state)).to(device)
        action_batch = torch.LongTensor(batch.action).to(device)
        reward_batch = torch.FloatTensor(batch.reward).to(device)
        next_state_batch = torch.FloatTensor(np.array(batch.next_state)).to(device)
        done_batch = torch.FloatTensor(batch.done).to(device)

        # Current Q-values: Q(s, a; theta)
        current_q = self.policy_net(state_batch).gather(1, action_batch.unsqueeze(1)).squeeze(1)

        # Target Q-values: r + gamma * max_a' Q(s', a'; theta^-)
        with torch.no_grad():
            next_q = self.target_net(next_state_batch).max(dim=1)[0]
            target_q = reward_batch + self.gamma * next_q * (1 - done_batch)

        # Huber loss (smooth L1) for robustness
        loss = F.smooth_l1_loss(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), max_norm=10.0)
        self.optimizer.step()

        self.losses.append(loss.item())
        return loss.item()

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())


print("DQNAgent class defined successfully.")

---

## 8. Training Loop

In [ ]:
def train_dqn(n_episodes=300, max_steps=10, image_size=64):
    """Train DQN agent on image processing environment."""
    env = ImageProcessingEnv(image_size=image_size, max_steps=max_steps)
    agent = DQNAgent(
        n_actions=env.n_actions,
        image_size=image_size,
        lr=3e-4,
        gamma=0.95,
        eps_start=1.0,
        eps_end=0.05,
        eps_decay=1500,
        buffer_size=20000,
        batch_size=64,
        target_update=50
    )

    episode_rewards = []
    episode_psnrs = []
    episode_actions = []

    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        actions_taken = []

        for step in range(max_steps):
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)

            agent.memory.push(state, action, reward, next_state, float(done))
            agent.optimize()

            state = next_state
            total_reward += reward
            actions_taken.append(action)

            if done:
                break

        # Update target network periodically
        if episode % agent.target_update == 0:
            agent.update_target()

        episode_rewards.append(total_reward)
        episode_psnrs.append(info['psnr'])
        episode_actions.append(actions_taken)

        if (episode + 1) % 50 == 0:
            avg_reward = np.mean(episode_rewards[-50:])
            avg_psnr = np.mean(episode_psnrs[-50:])
            eps = agent.get_epsilon()
            print(f"Episode {episode+1:4d} | Avg Reward: {avg_reward:7.3f} | "
                  f"Avg PSNR: {avg_psnr:.2f} dB | Epsilon: {eps:.3f}")

    return agent, episode_rewards, episode_psnrs, episode_actions, env


# Train the agent
agent, rewards, psnrs, actions_hist, env = train_dqn(n_episodes=300)

In [ ]:
# Comprehensive training visualization

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Episode Rewards
ax = axes[0, 0]
ax.plot(rewards, alpha=0.3, color='blue')
window = 20
if len(rewards) >= window:
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(rewards)), smoothed, color='blue', linewidth=2, label=f'{window}-ep moving avg')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('Episode Rewards (PSNR Improvement)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Training Loss
ax = axes[0, 1]
if agent.losses:
    losses_arr = np.array(agent.losses)
    ax.plot(losses_arr, alpha=0.2, color='red')
    window_l = min(100, len(losses_arr))
    if len(losses_arr) >= window_l:
        smoothed_loss = np.convolve(losses_arr, np.ones(window_l)/window_l, mode='valid')
        ax.plot(range(window_l-1, len(losses_arr)), smoothed_loss, color='red', linewidth=2)
ax.set_xlabel('Optimization Steps')
ax.set_ylabel('Loss (Smooth L1)')
ax.set_title('DQN Training Loss', fontsize=12)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# 3. PSNR Progression
ax = axes[1, 0]
ax.plot(psnrs, alpha=0.4, color='green')
if len(psnrs) >= window:
    smoothed_psnr = np.convolve(psnrs, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(psnrs)), smoothed_psnr, color='green', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Final PSNR (dB)')
ax.set_title('Image Quality (PSNR) Over Training', fontsize=12)
ax.grid(True, alpha=0.3)

# 4. Epsilon Decay
ax = axes[1, 1]
ax.plot(agent.epsilons, color='purple', linewidth=1.5)
ax.set_xlabel('Total Steps')
ax.set_ylabel('$\\epsilon$')
ax.set_title('Exploration Rate ($\\epsilon$) Decay', fontsize=12)
ax.grid(True, alpha=0.3)

plt.suptitle('DQN Training Dashboard — Image Processing Agent', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize learned action distribution

# Collect action frequencies from last 50 episodes
last_actions = [a for ep in actions_hist[-50:] for a in ep]
action_counts = np.bincount(last_actions, minlength=env.n_actions)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Action distribution
ax = axes[0]
bars = ax.bar(range(env.n_actions), action_counts / action_counts.sum(), color='steelblue', edgecolor='black')
ax.set_xticks(range(env.n_actions))
ax.set_xticklabels(env.ACTIONS, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Frequency')
ax.set_title('Learned Action Distribution (Last 50 Episodes)', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# Q-value evolution
ax = axes[1]
if agent.q_values_history:
    q_hist = np.array(agent.q_values_history)
    for i in range(min(env.n_actions, q_hist.shape[1])):
        ax.plot(q_hist[:, i], alpha=0.7, label=env.ACTIONS[i], linewidth=1)
    ax.set_xlabel('Greedy Steps')
    ax.set_ylabel('Q-value')
    ax.set_title('Q-Value Evolution During Training', fontsize=12)
    ax.legend(fontsize=8, ncol=2, loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate trained agent on a new image

def evaluate_agent(agent, env, n_eval=5):
    """Evaluate the trained agent and visualize results."""
    fig, axes = plt.subplots(n_eval, 3, figsize=(12, 4 * n_eval))

    for i in range(n_eval):
        state = env.reset()
        initial_psnr = env.prev_psnr
        initial_img = state.copy()

        for step in range(env.max_steps):
            with torch.no_grad():
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
                q_values = agent.policy_net(state_tensor)
                action = q_values.argmax(dim=1).item()
            state, reward, done, info = env.step(action)
            if done:
                break

        axes[i, 0].imshow(env.target_image.transpose(1, 2, 0))
        axes[i, 0].set_title('Target', fontsize=10)
        axes[i, 1].imshow(initial_img.transpose(1, 2, 0))
        axes[i, 1].set_title(f'Input (PSNR: {initial_psnr:.1f} dB)', fontsize=10)
        axes[i, 2].imshow(state.transpose(1, 2, 0))
        axes[i, 2].set_title(f'Output (PSNR: {info["psnr"]:.1f} dB)', fontsize=10)

        for ax in axes[i]:
            ax.axis('off')

    plt.suptitle('DQN Agent: Image Enhancement Results', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


evaluate_agent(agent, env, n_eval=3)

---

## 9. Summary

### Key Takeaways

| Component | Purpose | Mathematical Basis |
|-----------|---------|-------------------|
| **CNN Backbone** | Extract spatial features from image states | Hierarchical feature extraction |
| **Experience Replay** | Break temporal correlations | i.i.d. sampling for SGD convergence |
| **Target Network** | Stabilize training targets | $\gamma$-contraction of Bellman operator |
| **$\epsilon$-Greedy** | Explore action space | GLIE convergence guarantee |
| **Huber Loss** | Robust to reward outliers | Smooth L1: $\ell_\delta(a) = \begin{cases} \frac{1}{2}a^2 & |a| \leq \delta \\ \delta(|a| - \frac{1}{2}\delta) & \text{otherwise} \end{cases}$ |

### DQN Loss Function (Complete)

$$L(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{U}(\mathcal{D})}\left[\ell_\delta\left(r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta)\right)\right]$$

### Next Steps

- **Module 6.2**: Policy Gradient methods (REINFORCE) for continuous action spaces
- **Module 6.5**: Advanced tricks — prioritized replay, double DQN, dueling architecture